# Chapter 26: Pricing an Excess-of-Loss Cover

Code of Chapter 26 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

The walkthrough comes first, because it loads the data and imports used by the other examples of the chapter.

## 26.8 Python Walkthrough

In [ ]:
import numpy as np
import pandas as pd
from gitriangle import synthetic_history, exposure_by_accident_year
from gidevelop import average_factors, cumulative_factors, develop
from gilossmodels import fit_mle
from gicoverage import lev
from gireins import (ceded_triangle, burning_cost, exposure_rate, simulate_layers,
                     simulate_layer_years, xl_with_reinstatements, cat_event_set, cat_layer_cost)

policies, claims, tx = synthetic_history()
years = range(2021, 2026)
exposure = exposure_by_accident_year(policies, years)["earned_exposure"]

# Section 26.1.3: trended (to 2026) and developed burning cost
k = pd.Series([1.04 ** (2026 - y) for y in years], index=list(years))
tt = ceded_triangle(claims, tx, "incurred", 25_000, 25_000, 2021, 2025,
                    trend=0.04, base_year=2026).mul(k, axis=0)
ult = develop(tt, cumulative_factors(average_factors(tt, "volume"), 1.0))["ultimate"]
ult.sum() / exposure.sum()                                         # 32.01
bc = burning_cost(claims, 25_000, 25_000, 0.04, 2026.5, years, exposure)  # true ultimates
bc["layer"].sum() / exposure.sum()                                 # 35.17

# Section 26.2.2: exposure rating
P = {"alpha": 1.5, "theta": 50_000}
ilf = lambda u: lev("pareto", P, u)
sum(0.65 * sp * exposure_rate(ilf, pl, 250_000, 250_000)
    for pl, sp in {250_000: 2e6, 500_000: 3e6, 1_000_000: 1.5e6}.items())  # 431099

# Section 26.3: simulation by layer
x25 = claims[claims.accident_date.dt.year == 2025].ultimate_amount.to_numpy()
L = fit_mle(x25, "lognormal")
mu, s = float(L["mu"]) + np.log(1.04), float(L["sigma"])
sampler = lambda rng, n: rng.lognormal(mu, s, n)
sim = simulate_layers(654 * 1.05, sampler, [(25_000, 25_000), (50_000, 50_000), (100_000, 100_000)],
                      20_000, seed=26)
sim.agg(["mean", "std"]).round(0)

# Section 26.4.2: one reinstatement, with and without an AAD
yrs = simulate_layer_years(654 * 1.05, sampler, 20_000, seed=26)
R = xl_with_reinstatements(yrs, 50_000, 50_000, 1, aad=0)
(R.mean() / 0.8) / (1 + np.minimum(R, 50_000).mean() / 50_000)     # 51208

# Section 26.5.2: catastrophe layer
cat_layer_cost(cat_event_set(20), 250_000, 500_000)                # 23514